# Silver Layer Implimentation

In [0]:
# IMPORT
from pyspark.sql.functions import coalesce, to_timestamp, col, when, lit, regexp_replace

In [0]:
%sql
-- CREATING SCHEMA
USE CATALOG fixing_the_broken_data_pipeline;
CREATE SCHEMA IF NOT EXISTS silver;

## Vendor A Data Processing

In [0]:
df = spark.table("bronze.orders_vendor_a_shopify")

df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- product_sku: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- promo_code: string (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- loyalty_points: double (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_vendor: string (nullable = true)



In [0]:
display(df.limit(5))

order_id,order_date,customer_email,product_sku,quantity,unit_price,total_amount,payment_method,store_id,email,promo_code,discount_pct,loyalty_points,_source_file,_ingestion_timestamp,_source_vendor
US665523,03/17/2025,null,GADGET-03,4,31.22,124.88,apple_pay,STORE_120,jane.brown4226@outlook.com,null,null,null,s3a://vendor-data-24042026/vendor_a_shopify_data/orders_20250317_hour13.csv,2026-05-01T04:04:12.938Z,Vendor A
US655658,03/17/2025,null,GADGET-33,2,242.02,484.04,google_pay,STORE_103,alex.williams2225@yahoo.com,null,null,null,s3a://vendor-data-24042026/vendor_a_shopify_data/orders_20250317_hour13.csv,2026-05-01T04:04:12.938Z,Vendor A
US667513,03/17/2025,null,GADGET-20,3,107.14,321.42,credit_card,STORE_114,sam.jones3707@outlook.com,null,null,null,s3a://vendor-data-24042026/vendor_a_shopify_data/orders_20250317_hour13.csv,2026-05-01T04:04:12.938Z,Vendor A
US661295,03/17/2025,null,GADGET-61,2,294.92,589.84,bank_transfer,STORE_112,maria.brown6075@gmail.com,null,null,null,s3a://vendor-data-24042026/vendor_a_shopify_data/orders_20250317_hour13.csv,2026-05-01T04:04:12.938Z,Vendor A
US663081,03/17/2025,null,GADGET-46,2,87.96,175.92,google_pay,STORE_113,sam.smith3268@outlook.com,null,null,null,s3a://vendor-data-24042026/vendor_a_shopify_data/orders_20250317_hour13.csv,2026-05-01T04:04:12.938Z,Vendor A


In [0]:
display(df.where(col("email").isNotNull()).limit(5))

order_id,order_date,customer_email,product_sku,quantity,unit_price,total_amount,payment_method,store_id,email,promo_code,discount_pct,loyalty_points,_source_file,_ingestion_timestamp,_source_vendor
US432717,03/01/2025,null,GADGET-17,1,157.32,117.99,google_pay,STORE_115,john.smith2792@outlook.com,VIP25,25.0,250.0,s3a://vendor-data-24042026/vendor_a_shopify_data/orders_20250301_hour00.csv,2026-05-01T04:04:12.938Z,Vendor A
US431449,03/01/2025,null,GADGET-27,1,281.32,281.32,google_pay,STORE_119,jane.williams7207@gmail.com,null,null,null,s3a://vendor-data-24042026/vendor_a_shopify_data/orders_20250301_hour00.csv,2026-05-01T04:04:12.938Z,Vendor A
US440826,03/01/2025,null,GADGET-64,3,136.74,328.18,bank_transfer,STORE_101,alex.miller723@yahoo.com,SAVE20,20.0,200.0,s3a://vendor-data-24042026/vendor_a_shopify_data/orders_20250301_hour00.csv,2026-05-01T04:04:12.938Z,Vendor A
US440545,03/01/2025,null,GADGET-18,2,281.2,506.16,bank_transfer,STORE_119,lisa.brown9943@outlook.com,SAVE10,10.0,100.0,s3a://vendor-data-24042026/vendor_a_shopify_data/orders_20250301_hour00.csv,2026-05-01T04:04:12.938Z,Vendor A
US437868,03/01/2025,null,GADGET-49,3,282.35,847.05,google_pay,STORE_111,lisa.williams48@yahoo.com,null,null,null,s3a://vendor-data-24042026/vendor_a_shopify_data/orders_20250301_hour00.csv,2026-05-01T04:04:12.938Z,Vendor A


- when `email` is not NULL, `customer_email` is Null

In [0]:
def normalize_to_silver(df):
    # MERGING THE EMail AND CUSTOMER_EMaIL
    df = df.withColumn(
        "customer_email",
        coalesce(col("email"), col("customer_email"))
    )

    # CHANGING ORDER DATE DATATYPE
    df = df.withColumn(
        "order_date",
        to_timestamp(col("order_date"), "MM/dd/yyyy")
    )

    # add line number as default 1
    df = df.withColumn("line_number", lit(1))

    # renaming the columns unit_price and total_amount as unit_price_usd and total_amount_usd
    df = df.withColumnRenamed("unit_price", "unit_price_usd")
    df = df.withColumnRenamed("total_amount", "total_amount_usd")

    # discount amount
    df = df.withColumn(
        "discount_amount_usd",
        when(
            col("discount_pct").isNotNull(),
                col("unit_price_usd") * col("quantity") * col("discount_pct") / 100
        ).otherwise(lit(None))
    )

    # payment method standardization+
    df = df.withColumn(
        "payment_method",
        when(
            col("payment_method") == "apple pay", lit("mobile_wallet")
        ).when(
            col("payment_method") == "google pay", lit("mobile_wallet")
        ).otherwise(col("payment_method"))
    )

    # adding country and currency metadata
    df = df.withColumn("country_code", lit("US"))
    df = df.withColumn("source_currency", lit("USD"))
    df = df.withColumn("exchanmge_rate", lit(1.0).cast("decimal(10, 6)"))
    df = df.withColumn("vat_amount_usd", lit(None).cast("decimal(10, 2)"))

    return df.select(
        col("order_id"), col("line_number"), col("order_date"), col("customer_email"), col("product_sku"), col("quantity"), col("unit_price_usd"), col("total_amount_usd"), col("discount_amount_usd"), col("payment_method"), col("country_code"), col("promo_code"),
        col("loyalty_points"), col("_source_vendor"), col("source_currency"), col("exchanmge_rate"), col("vat_amount_usd"),
        col("_ingestion_timestamp"), col("_source_file")
    )

## Vendor B Data Processing